# LegalQA Task 2 — Kaggle Dual-T4 CUDA Smoke Gate (Notion DSC 2026)
Lightweight CUDA, VRAM, QLoRA, and Liger-kernel compatibility smoke test launcher.
- **Target Hardware**: Kaggle Dual NVIDIA T4 (GPU 0: Generator | GPU 1: Retrieval/Reranker)
- **Goal**: Validate worst-case sequence probe, short endurance stability, and mini-evaluation before Colab A100 training.
- **Artifact Export**: Outputs  to .

In [ ]:
# Cell 1: Environment & Guardrails
import os, sys

# Disable Transformers 5.0 async load to prevent transient VRAM spikes during 4-bit load on T4
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
print("Transformers async model loading: DISABLED for T4-safe QLoRA load")

SEED = 42
CONFIG_PATH = "configs/kaggle_smoke_t4.yaml"
print(f"Config target: {CONFIG_PATH}")


In [ ]:
# Cell 2: Hardware & Device Allocation
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Kaggle GPU execution but torch.cuda.is_available() is False.")

gpu_count = torch.cuda.device_count()
print(f"CUDA GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB | Compute: sm_{p.major}{p.minor}")

GEN_DEVICE = "cuda:0"
RETRIEVAL_DEVICE = "cuda:1" if gpu_count >= 2 else "cuda:0"
print(f"Hardware Allocation -> Generator: {GEN_DEVICE} | Retrieval/Reranker: {RETRIEVAL_DEVICE}")


In [ ]:
# Cell 3: Code Root & Workspace Bootstrap
from pathlib import Path
import os, sys

candidate_roots = [
    os.path.abspath("."),
    os.path.abspath("LegalQA"),
    os.path.abspath("/kaggle/working"),
    os.path.abspath("/kaggle/working/LegalQA")
]
code_root = next((r for r in candidate_roots if os.path.isdir(os.path.join(r, "src", "task2"))), None)
if not code_root:
    print("Cloning LegalQA repository from GitHub...")
    os.system("git clone https://github.com/silent9669/LegalQA.git")
    code_root = os.path.abspath("LegalQA")

if code_root not in sys.path:
    sys.path.insert(0, code_root)
print(f"Active Code Root: {code_root}")


In [ ]:
# Cell 4: Dataset Mount & Integrity Verification
from src.task2.path_resolver import resolve_runtime_paths
from src.task2.dataset.validator import validate_dataset

paths = resolve_runtime_paths("/kaggle/input", strict=False)
print(f"Resolved Dataset Root: {paths.get('runtime_root')}")

schema_file = os.path.join(code_root, "configs/dataset_schema.yaml")
val_report = validate_dataset(data_dir=paths["runtime_root"], schema_path=schema_file)
print(f"Dataset Manifest Verified: {val_report.get('manifest_verified')} (Status: {val_report.get('status')})")
if val_report.get("status") != "PASS":
    raise RuntimeError(f"Dataset validation failed: {val_report.get('errors')}")


In [ ]:
# Cell 5: Execute Smoke Profile Runner
from src.task2.pipeline.profiles import load_profile_from_yaml
from src.task2.pipeline.runner import run_pipeline

active_profile = load_profile_from_yaml(os.path.join(code_root, CONFIG_PATH))
print(f"Active Smoke Profile: {active_profile.name}")

paths["public_test_path"] = os.path.join(paths["data_dir"], "public-official.json")
if not os.path.exists(paths["public_test_path"]):
    paths["public_test_path"] = os.path.join(paths["runtime_root"], "public-official.json")

pipeline_outputs = run_pipeline(
    profile=active_profile,
    paths=paths,
    gen_device=GEN_DEVICE,
    retrieval_device=RETRIEVAL_DEVICE,
    output_dir="/kaggle/working",
    seed=SEED,
    code_root=code_root,
    allow_single_gpu=True,
)
print("
Smoke execution completed successfully.")


In [ ]:
# Cell 6: Export Evidence Bundle
import json
report_path = "/kaggle/working/kaggle_smoke_report.json"
smoke_summary = {
    "status": "PASS",
    "profile": active_profile.name,
    "hardware": {
        "gpu_count": gpu_count,
        "gen_device": GEN_DEVICE,
        "retrieval_device": RETRIEVAL_DEVICE,
    },
    "dataset_verified": val_report.get("manifest_verified"),
    "stages_executed": list(pipeline_outputs.get("stages", {}).keys()) if isinstance(pipeline_outputs, dict) else [],
}
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(smoke_summary, f, indent=2)
print(f"Saved smoke gate verification report to {report_path}")
